# Fine-tune Qwen3-Embedding-0.6B on CodeSearchNet

Contrastive fine-tuning of `Qwen/Qwen3-Embedding-0.6B` using the train split of CodeSearchNet.
Uses LoRA to fit within Colab T4 (15 GB VRAM / 13 GB RAM).

Checkpoints are saved after every N steps so you can resume if the runtime dies.

**Runtime → Change runtime type → T4 GPU → Run all.**

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q transformers>=4.44.0 datasets peft accelerate torch sentence-transformers
!pip uninstall -y torchao  # incompatible with Colab's PEFT, not needed here

In [ ]:
import json
import os
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from datasets import load_dataset
from transformers import AutoModel, AutoTokenizer, get_cosine_schedule_with_warmup
from peft import LoraConfig, get_peft_model, TaskType

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

## 1. Load CodeSearchNet train split

In [ ]:
# How many examples to use for training (across all languages).
# 10 000 is a good starting point for Colab; increase if you have time.
MAX_TRAIN_EXAMPLES = 80_000
LANGUAGES = ["python", "java", "javascript", "go", "ruby", "php"]

per_lang_limit = MAX_TRAIN_EXAMPLES // len(LANGUAGES)

train_pairs = []  # list of (query, code) strings

for lang in LANGUAGES:
    print(f"Loading {lang} ...")
    ds = load_dataset("code-search-net/code_search_net", lang, split="train")
    # Take the *last* examples instead of the first (shuffled order differs)
    subset = ds.select(range(max(0, len(ds) - per_lang_limit), len(ds)))
    count = 0
    for example in subset:
        code = (example.get("func_code_string") or "").strip()
        doc = (example.get("func_documentation_string") or "").strip()
        if code and doc:
            train_pairs.append((doc, code))
            count += 1
    print(f"  collected {count} pairs from {lang}")

print(f"\nTotal training pairs: {len(train_pairs)}")
print(f"Example query: {train_pairs[0][0][:120]}...")
print(f"Example code:  {train_pairs[0][1][:120]}...")

## 2. Dataset & DataLoader

In [ ]:
QUERY_INSTRUCTION = "Retrieve relevant source code based on the user query"
MAX_SEQ_LENGTH = 512


class CodeSearchPairDataset(Dataset):
    """Simple (query, code) pair dataset."""

    def __init__(self, pairs):
        self.pairs = pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        query, code = self.pairs[idx]
        return {"query": query, "code": code}


dataset = CodeSearchPairDataset(train_pairs)

# 90/10 train/val split
val_size = max(1, int(0.1 * len(dataset)))
train_size = len(dataset) - val_size
train_dataset, val_dataset = torch.utils.data.random_split(
    dataset, [train_size, val_size], generator=torch.Generator().manual_seed(SEED)
)
print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}")

## 3. Load model & apply LoRA

In [ ]:
MODEL_NAME = "Qwen/Qwen3-Embedding-0.6B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModel.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    trust_remote_code=True,
).to(DEVICE)

# Apply LoRA to projection layers (lightweight, saves VRAM)
lora_config = LoraConfig(
    task_type=TaskType.FEATURE_EXTRACTION,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

EMBED_DIM = model.config.hidden_size
print(f"Embedding dimension: {EMBED_DIM}")

## 4. Last-token pooling (same as your embedder.py)

In [ ]:
def last_token_pool(last_hidden_states: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
    left_padding = attention_mask[:, -1].sum() == attention_mask.shape[0]
    if left_padding:
        return last_hidden_states[:, -1]
    seq_lengths = attention_mask.sum(dim=1) - 1
    batch_size = last_hidden_states.shape[0]
    return last_hidden_states[
        torch.arange(batch_size, device=last_hidden_states.device),
        seq_lengths,
    ]


def encode_texts(model, tokenizer, texts, is_query=False, batch_size=128, max_length=512):
    """Encode a list of texts into L2-normalised embeddings."""
    model.eval()
    all_embs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        if is_query:
            batch = [f"Instruct: {QUERY_INSTRUCTION}\nQuery: {t}" for t in batch]
        encoded = tokenizer(
            batch, padding=True, truncation=True,
            max_length=max_length, return_tensors="pt",
        ).to(model.device)
        with torch.no_grad():
            out = model(**encoded)
        embs = last_token_pool(out.last_hidden_state, encoded["attention_mask"])
        embs = F.normalize(embs, p=2, dim=1)
        all_embs.append(embs.cpu().float())
    return torch.vstack(all_embs)

## 5. Contrastive training loop with checkpoints

In [ ]:
# ── Hyperparameters ───────────────────────────────────────────────
# 0.6B model fits ~256 on T4 without LoRA; with LoRA we have even more headroom.
BATCH_SIZE = 64
GRADIENT_ACCUM_STEPS = 2
EFFECTIVE_BATCH = BATCH_SIZE * GRADIENT_ACCUM_STEPS
EPOCHS = 2
LR = 2e-5
WARMUP_RATIO = 0.1
MAX_STEPS = -1           # -1 = use all epochs
VAL_STEPS = 200          # validate every N optimiser steps
CHECKPOINT_STEPS = 200   # save checkpoint every N optimiser steps
CHECKPOINT_DIR = Path("checkpoints")
CHECKPOINT_DIR.mkdir(exist_ok=True)

# In-batch negatives temperature
TEMPERATURE = 0.02

print(f"Effective batch size: {EFFECTIVE_BATCH}")
print(f"Will checkpoint every {CHECKPOINT_STEPS} steps")

In [ ]:
def save_checkpoint(model, optimizer, scheduler, step, epoch, best_val_loss, tag="last"):
    """Save a full training checkpoint to disk."""
    ckpt_dir = CHECKPOINT_DIR / tag
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(str(ckpt_dir / "adapter"))
    tokenizer.save_pretrained(str(ckpt_dir / "adapter"))
    torch.save({
        "step": step,
        "epoch": epoch,
        "best_val_loss": best_val_loss,
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict() if scheduler else None,
    }, str(ckpt_dir / "training_state.pt"))
    print(f"  [checkpoint saved: {ckpt_dir}]")


def load_checkpoint(model, optimizer, scheduler):
    """Resume from the latest checkpoint if one exists.

    Returns (start_step, start_epoch, best_val_loss) or (0, 0, float('inf')).
    """
    last_dir = CHECKPOINT_DIR / "last"
    state_path = last_dir / "training_state.pt"
    adapter_path = last_dir / "adapter"
    if not state_path.exists() or not adapter_path.exists():
        print("No checkpoint found — starting from scratch.")
        return 0, 0, float("inf")

    print(f"Resuming from {last_dir} ...")
    model = get_peft_model(model, lora_config)
    model.load_adapter(str(adapter_path))
    state = torch.load(str(state_path), map_location=DEVICE)
    optimizer.load_state_dict(state["optimizer_state_dict"])
    if scheduler and state["scheduler_state_dict"]:
        scheduler.load_state_dict(state["scheduler_state_dict"])
    print(f"  Resumed from step {state['step']}, epoch {state['epoch']}")
    return state["step"], state["epoch"], state["best_val_loss"]

In [ ]:
def collate_fn(batch, tokenizer, max_length=512):
    """Tokenise query+code pairs for contrastive learning."""
    queries = [f"Instruct: {QUERY_INSTRUCTION}\nQuery: {item['query']}" for item in batch]
    codes = [item["code"] for item in batch]

    q_enc = tokenizer(
        queries, padding=True, truncation=True,
        max_length=max_length, return_tensors="pt",
    )
    c_enc = tokenizer(
        codes, padding=True, truncation=True,
        max_length=max_length, return_tensors="pt",
    )
    return {"q": q_enc, "c": c_enc}


def contrastive_loss(q_embs, c_embs, temperature=TEMPERATURE):
    """Symmetric InfoNCE / contrastive loss with in-batch negatives.

    Each query's positive is the code at the same index.
    All other codes in the batch serve as negatives.
    """
    # (B, D)
    q_embs = F.normalize(q_embs, p=2, dim=1)
    c_embs = F.normalize(c_embs, p=2, dim=1)
    # (B, B) similarity matrix
    logits = torch.matmul(q_embs, c_embs.T) / temperature
    labels = torch.arange(logits.size(0), device=logits.device)
    loss_q2c = F.cross_entropy(logits, labels)
    loss_c2q = F.cross_entropy(logits.T, labels)
    return (loss_q2c + loss_c2q) / 2


def validate(model, tokenizer, val_dl):
    """Compute average contrastive loss on the validation set."""
    model.eval()
    total_loss = 0.0
    n_batches = 0
    with torch.no_grad():
        for batch in val_dl:
            q_ids = batch["q"].to(model.device)
            c_ids = batch["c"].to(model.device)
            q_out = model(**q_ids)
            c_out = model(**c_ids)
            q_emb = last_token_pool(q_out.last_hidden_state, q_ids["attention_mask"])
            c_emb = last_token_pool(c_out.last_hidden_state, c_ids["attention_mask"])
            loss = contrastive_loss(q_emb, c_emb)
            total_loss += loss.item()
            n_batches += 1
    return total_loss / max(n_batches, 1)

In [ ]:
def train():
    model.train()

    train_dl = DataLoader(
        train_dataset, batch_size=BATCH_SIZE, shuffle=True,
        collate_fn=lambda b: collate_fn(b, tokenizer, MAX_SEQ_LENGTH),
        num_workers=4, pin_memory=True, drop_last=True,
    )
    val_dl = DataLoader(
        val_dataset, batch_size=BATCH_SIZE, shuffle=False,
        collate_fn=lambda b: collate_fn(b, tokenizer, MAX_SEQ_LENGTH),
        num_workers=2, pin_memory=True,
    )

    num_update_steps = (len(train_dl) * EPOCHS) // GRADIENT_ACCUM_STEPS
    if MAX_STEPS > 0:
        num_update_steps = min(num_update_steps, MAX_STEPS)
    num_warmup = int(num_update_steps * WARMUP_RATIO)

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    scheduler = get_cosine_schedule_with_warmup(
        optimizer, num_warmup_steps=num_warmup, num_training_steps=num_update_steps,
    )

    # ── Try to resume from checkpoint ──
    global_step, start_epoch, best_val_loss = load_checkpoint(model, optimizer, scheduler)

    print(f"\nStarting training: {num_update_steps} total optimiser steps, {num_warmup} warmup")
    print(f"Train size: {len(train_dataset)}, Val size: {len(val_dataset)}")

    scaler = torch.amp.GradScaler("cuda") if DEVICE == "cuda" else None

    step = global_step
    running_loss = 0.0
    micro_step = 0

    for epoch in range(start_epoch, EPOCHS):
        print(f"\n{'='*60}")
        print(f"Epoch {epoch + 1}/{EPOCHS}")
        print(f"{'='*60}")
        model.train()

        for batch in train_dl:
            q_ids = batch["q"].to(DEVICE)
            c_ids = batch["c"].to(DEVICE)

            with torch.amp.autocast("cuda", dtype=torch.float16) if DEVICE == "cuda" else torch.no_grad():
                q_out = model(**q_ids)
                c_out = model(**c_ids)
                q_emb = last_token_pool(q_out.last_hidden_state, q_ids["attention_mask"])
                c_emb = last_token_pool(c_out.last_hidden_state, c_ids["attention_mask"])
                loss = contrastive_loss(q_emb, c_emb) / GRADIENT_ACCUM_STEPS

            if scaler:
                scaler.scale(loss).backward()
            else:
                loss.backward()

            running_loss += loss.item() * GRADIENT_ACCUM_STEPS
            micro_step += 1

            if micro_step % GRADIENT_ACCUM_STEPS == 0:
                if scaler:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    optimizer.step()
                scheduler.step()
                optimizer.zero_grad()
                step += 1

                avg_loss = running_loss / GRADIENT_ACCUM_STEPS

                if step % 50 == 0:
                    lr_now = scheduler.get_last_lr()[0]
                    print(f"  step {step:>6d}/{num_update_steps}  loss={avg_loss:.4f}  lr={lr_now:.2e}")

                running_loss = 0.0

                # ── Validation ──
                if step % VAL_STEPS == 0:
                    val_loss = validate(model, tokenizer, val_dl)
                    print(f"  [val] step {step}  val_loss={val_loss:.4f}")
                    if val_loss < best_val_loss:
                        best_val_loss = val_loss
                        save_checkpoint(model, optimizer, scheduler, step, epoch, best_val_loss, tag="best")

                # ── Periodic checkpoint ──
                if step % CHECKPOINT_STEPS == 0:
                    save_checkpoint(model, optimizer, scheduler, step, epoch, best_val_loss, tag="last")

                if MAX_STEPS > 0 and step >= MAX_STEPS:
                    print(f"Reached MAX_STEPS={MAX_STEPS}, stopping.")
                    break

        if MAX_STEPS > 0 and step >= MAX_STEPS:
            break

    # ── Final save ──
    save_checkpoint(model, optimizer, scheduler, step, epoch, best_val_loss, tag="last")

    # Also save the best as a standalone adapter for easy loading
    best_dir = CHECKPOINT_DIR / "best"
    if best_dir.exists():
        final_dir = Path("qwen3-embed-0.6b-finetuned")
        final_dir.mkdir(exist_ok=True)
        # Copy best adapter weights
        import shutil
        shutil.copytree(str(best_dir / "adapter"), str(final_dir), dirs_exist_ok=True)
        print(f"\nBest model saved to {final_dir}/")

    print(f"\nTraining complete. Best val loss: {best_val_loss:.4f}")
    return model

In [ ]:
model = train()

## 6. Quick sanity check — compare base vs fine-tuned

In [ ]:
test_queries = [
    "read a file and return its contents",
    "sort a list of integers in ascending order",
    "make an HTTP GET request and return the response body",
]
test_codes = [
    "def read_file(path):\n    with open(path) as f:\n        return f.read()",
    "def sort_list(lst):\n    return sorted(lst)",
    "import requests\ndef get(url):\n    return requests.get(url).text",
]

# Load base model fresh for comparison
base_model_eval = AutoModel.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16, trust_remote_code=True,
).to(DEVICE)
base_model_eval.eval()

q_embs_base = encode_texts(base_model_eval, tokenizer, test_queries, is_query=True)
c_embs_base = encode_texts(base_model_eval, tokenizer, test_codes, is_query=False)
sim_base = torch.matmul(q_embs_base, c_embs_base.T).diag()

q_embs_ft = encode_texts(model, tokenizer, test_queries, is_query=True)
c_embs_ft = encode_texts(model, tokenizer, test_codes, is_query=False)
sim_ft = torch.matmul(q_embs_ft, c_embs_ft.T).diag()

del base_model_eval
torch.cuda.empty_cache()

print(f"{'Query':<55} {'Base':>6} {'FT':>6} {'Delta':>7}")
print("-" * 80)
for i, q in enumerate(test_queries):
    print(f"{q[:53]:<55} {sim_base[i]:.4f} {sim_ft[i]:.4f} {sim_ft[i]-sim_base[i]:+.4f}")

## 7. Export for use with your search engine

The fine-tuned adapter is in `checkpoints/best/adapter/`. You can merge it into the base model and push to HuggingFace, or load it directly with PEFT:

```python
from peft import PeftModel
from transformers import AutoModel

base = AutoModel.from_pretrained("Qwen/Qwen3-Embedding-0.6B", torch_dtype=torch.float16)
model = PeftModel.from_pretrained(base, "checkpoints/best/adapter")
model = model.merge_and_unload()  # merge LoRA weights into base
model.save_pretrained("qwen3-embed-0.6b-finetuned")
```

Then point your `example_config.yaml` at `qwen3-embed-0.6b-finetuned`.

In [ ]:
# List saved checkpoints
for p in sorted(CHECKPOINT_DIR.rglob("*")):
    if p.is_file():
        print(f"{p}  ({p.stat().st_size / 1024**2:.1f} MB)")